In [ ]:
!pip install -q datasets nltk langdetect scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 13.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [ ]:
# ============================================================
# COURSERA NLP PREPROCESSING PIPELINE
# Data : azrai99/coursera-course-dataset (HuggingFace)
# Actual columns: title, Description, Skills, Level,
#                 rating, num_reviews, Instructor, Organization
# ============================================================

# ------------------------------------------------------------
# IMPORTS
# ------------------------------------------------------------
import ast
import logging
import re
import unicodedata

import nltk
import numpy as np
import pandas as pd
from datasets import load_dataset as hf_load_dataset
from langdetect import detect, LangDetectException
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

for _r in ["stopwords", "wordnet", "omw-1.4"]:
    nltk.download(_r, quiet=True)

# ------------------------------------------------------------
# LOGGING
# ------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger(__name__)

# ------------------------------------------------------------
# GLOBAL CONFIGURATION  (edit as needed)
# ------------------------------------------------------------
DATASET_NAME: str           = "azrai99/coursera-course-dataset"
DATASET_SPLIT: str          = "train"
SAMPLE_SIZE: int            = 25
RANDOM_SEED: int            = 42
MIN_DESC_WORDS: int         = 20
MAX_DESC_WORDS: int         = 500
SIMILARITY_THRESHOLD: float = 0.90
OUTPUT_SAMPLE_CSV: str      = "coursera_clean_sample.csv"
OUTPUT_FULL_CSV: str        = "coursera_clean_full.csv"

# Actual column names in the dataset
COL_TITLE: str       = "title"
COL_DESC: str        = "Description"
COL_SKILLS: str      = "Skills"
COL_LEVEL: str       = "Level"
COL_RATING: str      = "rating"
COL_REVIEWS: str     = "num_reviews"

DIFFICULTY_MAP: dict = {
    "beginner": "Beginner", "begin": "Beginner", "easy": "Beginner",
    "introductory": "Beginner", "intro": "Beginner", "novice": "Beginner",
    "intermediate": "Intermediate", "inter": "Intermediate",
    "medium": "Intermediate", "moderate": "Intermediate",
    "advanced": "Advanced", "advance": "Advanced", "expert": "Advanced",
    "hard": "Advanced", "difficult": "Advanced",
    "mixed": "Mixed",
}

_STOP_WORDS: set               = set(stopwords.words("english"))
_LEMMATIZER: WordNetLemmatizer = WordNetLemmatizer()


# ============================================================
# STEP 1 - DATA LOADING
# ============================================================
def load_data(
    dataset_name: str = DATASET_NAME,
    split: str = DATASET_SPLIT,
) -> pd.DataFrame:
    """
    Load a HuggingFace dataset and return it as a pandas DataFrame.

    Parameters
    ----------
    dataset_name : str
        HuggingFace dataset identifier.
    split : str
        Dataset split to load.

    Returns
    -------
    pd.DataFrame
    """
    logger.info("Loading dataset '%s' (split='%s') ...", dataset_name, split)
    try:
        ds = hf_load_dataset(dataset_name, split=split)
        df = ds.to_pandas()
        logger.info("Dataset loaded -- %d rows, %d columns.", *df.shape)
        logger.info("Columns found: %s", df.columns.tolist())
        return df
    except Exception as exc:
        logger.error("Failed to load dataset: %s", exc)
        raise


# ============================================================
# STEP 2 - DATA INSPECTION
# ============================================================
def inspect_data(df: pd.DataFrame) -> None:
    """
    Log basic metadata and display a sample of the DataFrame.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame to inspect.
    """
    logger.info("Shape      : %s", df.shape)
    logger.info("Columns    : %s", df.columns.tolist())
    logger.info("Data types :\n%s", df.dtypes.to_string())
    logger.info("Null counts:\n%s", df.isnull().sum().to_string())


# ============================================================
# STEP 3 - DATA CLEANING
# ============================================================
def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Drop missing titles/descriptions, exact duplicates, and
    courses with extremely short descriptions.

    Parameters
    ----------
    df : pd.DataFrame

    Returns
    -------
    pd.DataFrame
    """

    initial = len(df)
    unnamed_cols = [c for c in df.columns if re.match(r"unnamed:\s*\d+", c, re.IGNORECASE)]
    if unnamed_cols:
        df = df.drop(columns=unnamed_cols)

    # Drop rows where title or description is null
    df = df.dropna(subset=[COL_TITLE, COL_DESC])
    df = df[
        df[COL_TITLE].str.strip().astype(bool)
        & df[COL_DESC].str.strip().astype(bool)
    ]
    logger.info("Dropped %d rows with missing title/description.", initial - len(df))

    # Exact duplicates
    before_dedup = len(df)
    df = df.drop_duplicates(subset=[COL_TITLE, COL_DESC], keep="first")
    logger.info("Dropped %d exact duplicate rows.", before_dedup - len(df))

    # Short descriptions
    before_short = len(df)
    df = df[df[COL_DESC].str.split().str.len() >= MIN_DESC_WORDS]
    logger.info(
        "Dropped %d rows with descriptions shorter than %d words.",
        before_short - len(df), MIN_DESC_WORDS,
    )

    df = df.reset_index(drop=True)
    logger.info("Cleaning complete -- %d rows remain.", len(df))
    return df


# ============================================================
# STEP 4 - TEXT PREPROCESSING
# ============================================================
def _clean_text(text: str, remove_stopwords: bool = False) -> str:
    if not isinstance(text, str):
        return ""
    text = text.strip()

    _COURSERA_UNIT = re.compile(
        r"\d+\s*(?:video[s]?|reading[s]?|quiz(?:zes)?"
        r"|programming\s+assignment[s]?|assignment[s]?"
        r"|peer\s+review[s]?|discussion\s+prompt[s]?|discussion\s+forum[s]?"
        r"|hour[s]?|minute[s]?|module[s]?|week[s]?|plugin[s]?)",
        re.IGNORECASE,
    )
    # Fix stuck tokens: 'video4' -> 'video 4', '1reading' -> '1 reading'
    text = re.sub(r"([a-zA-Z])(\d)", r"\1 \2", text)
    text = re.sub(r"(\d)([a-zA-Z])", r"\1 \2", text)
    # Remove boilerplate at the sentence level:
    # Split on sentence boundaries, drop any sentence that is entirely
    # composed of Coursera module-summary tokens (no real words remain),
    # then rejoin the surviving sentences.
    sentences = re.split(r"(?<=[.!?])\s+", text)
    kept = []
    for sent in sentences:
        masked = _COURSERA_UNIT.sub("", sent)
        if re.search(r"[a-zA-Z]", masked):   # real words survive -> keep
            kept.append(sent)
        # else: pure boilerplate sentence -> discard
    text = " ".join(kept) if kept else text   # fallback: keep original if all dropped
    # Also strip any leftover unit tokens not inside sentence boundaries
    prev = None
    while prev != text:
        prev = text
        text = _COURSERA_UNIT.sub("\x00", text)
        text = re.sub(r"(\x00\s*){2,}", " ", text)
        text = re.sub(r"\x00", "", text)

    text = text.lower()
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = unicodedata.normalize("NFKD", text)
    text = text.encode("ascii", "ignore").decode("ascii")
    text = re.sub(r"[^a-z0-9\.\s]", " ", text)
    text = re.sub(r"(?<![a-z0-9])\.|\\.(?![a-z])", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    if remove_stopwords:
        tokens = [t for t in text.split() if t not in _STOP_WORDS]
        tokens = [_LEMMATIZER.lemmatize(t) for t in tokens]
        text = " ".join(tokens)
    return text


def preprocess_text(df: pd.DataFrame) -> pd.DataFrame:
    """
    Apply _clean_text to title, Description, and Skills.
    Results stored in title_clean, description_clean, skills_clean.

    Parameters
    ----------
    df : pd.DataFrame

    Returns
    -------
    pd.DataFrame
    """
    logger.info("Preprocessing text columns ...")
    df = df.copy()
    df[COL_TITLE] = df[COL_TITLE].apply(_clean_text)
    df[COL_DESC]  = df[COL_DESC].apply(_clean_text)
    if COL_SKILLS in df.columns:
        df[COL_SKILLS] = df[COL_SKILLS].apply(
            lambda x: _clean_text(str(x)) if pd.notna(x) else ""
        )
    logger.info("Text preprocessing complete.")
    return df


# ============================================================
# STEP 5 - FEATURE NORMALISATION
# ============================================================
def _parse_skills(raw: object) -> list:
    """
    Parse the Skills field into a deduplicated Python list.

    Parameters
    ----------
    raw : object

    Returns
    -------
    list
    """
    if isinstance(raw, list):
        skills = raw
    elif isinstance(raw, str):
        raw = raw.strip()
        if raw.startswith("["):
            try:
                skills = ast.literal_eval(raw)
            except (ValueError, SyntaxError):
                skills = [s.strip() for s in raw.strip("[]").split(",")]
        else:
            skills = [s.strip() for s in raw.split(",")]
    else:
        return []
    seen: set = set()
    result: list = []
    for s in skills:
        key = str(s).strip().lower()
        if key and key not in seen:
            seen.add(key)
            result.append(str(s).strip())
    return result


def _normalise_difficulty(raw: object) -> str:
    """
    Map a raw difficulty label to a canonical value.

    Parameters
    ----------
    raw : object

    Returns
    -------
    str -- one of Beginner / Intermediate / Advanced / Mixed / Unknown
    """
    if not isinstance(raw, str) or not raw.strip():
        return "Unknown"
    key = raw.strip().lower()
    for fragment, canonical in DIFFICULTY_MAP.items():
        if fragment in key:
            return canonical
    return "Unknown"


def normalize_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Normalise Level, rating, num_reviews, and Skills columns.

    Parameters
    ----------
    df : pd.DataFrame

    Returns
    -------
    pd.DataFrame
    """
    logger.info("Normalising features ...")
    df = df.copy()

    if COL_LEVEL in df.columns:
        df[COL_LEVEL] = df[COL_LEVEL].apply(_normalise_difficulty)

    if COL_RATING in df.columns:
        df[COL_RATING] = pd.to_numeric(df[COL_RATING], errors="coerce")

    if COL_REVIEWS in df.columns:
        df[COL_REVIEWS] = (
            df[COL_REVIEWS].astype(str).str.replace(r"[^\d]", "", regex=True)
        )
        df[COL_REVIEWS] = pd.to_numeric(
            df[COL_REVIEWS], errors="coerce"
        ).astype("Int64")

    if COL_SKILLS in df.columns:
        df[COL_SKILLS] = df[COL_SKILLS].apply(_parse_skills)

    logger.info("Feature normalisation complete.")
    return df


# ============================================================
# STEP 6 - ADVANCED DEDUPLICATION
# ============================================================
def deduplicate_data(
    df: pd.DataFrame,
    threshold: float = SIMILARITY_THRESHOLD,
    text_col: str = COL_DESC,
) -> pd.DataFrame:
    """
    Remove near-duplicate courses via TF-IDF cosine similarity.
    Processed in batches of 500 to avoid memory overflow.

    Parameters
    ----------
    df : pd.DataFrame
    threshold : float
        Rows with cosine similarity above this value are dropped.
    text_col : str

    Returns
    -------
    pd.DataFrame
    """
    logger.info("Near-duplicate detection (threshold=%.2f) ...", threshold)
    col = text_col if text_col in df.columns else COL_DESC
    texts = df[col].fillna("").tolist()

    vectoriser = TfidfVectorizer(max_features=10_000, sublinear_tf=True)
    tfidf_matrix = vectoriser.fit_transform(texts)

    n = len(df)
    to_drop: set = set()

    for start in range(0, n, 500):
        end = min(start + 500, n)
        sim_matrix = cosine_similarity(tfidf_matrix[start:end], tfidf_matrix)
        for local_i, global_i in enumerate(range(start, end)):
            if global_i in to_drop:
                continue
            sim_row = sim_matrix[local_i, global_i + 1:]
            dups = np.where(sim_row > threshold)[0] + global_i + 1
            to_drop.update(int(d) for d in dups)

    before = len(df)
    df = df.drop(index=list(to_drop)).reset_index(drop=True)
    logger.info(
        "Near-deduplication removed %d rows -- %d remain.",
        before - len(df), len(df),
    )
    return df


# ============================================================
# STEP 7 - QUALITY FILTERING
# ============================================================
def _is_english(text: str) -> bool:
    """
    Return True if langdetect identifies the text as English.

    Parameters
    ----------
    text : str

    Returns
    -------
    bool
    """
    try:
        return detect(text) == "en"
    except LangDetectException:
        return False


def _is_noisy(text: str) -> bool:
    """
    Return True if the text is junk (low alpha ratio or
    pathological character repetition).

    Parameters
    ----------
    text : str

    Returns
    -------
    bool
    """
    if not isinstance(text, str) or len(text) < 10:
        return True
    if sum(c.isalpha() for c in text) / max(len(text), 1) < 0.5:
        return True
    if re.search(r"(.)\1{10,}", text):
        return True
    return False


def filter_quality(
    df: pd.DataFrame,
    filter_language: bool = True,
) -> pd.DataFrame:
    """
    Keep only English courses with clean descriptions within
    the configured word-count range.

    Parameters
    ----------
    df : pd.DataFrame
    filter_language : bool

    Returns
    -------
    pd.DataFrame
    """
    logger.info("Applying quality filters ...")
    before = len(df)

    wc = df[COL_DESC].str.split().str.len()
    df = df[(wc >= MIN_DESC_WORDS) & (wc <= MAX_DESC_WORDS)]
    logger.info("Word-count filter: %d -> %d rows.", before, len(df))

    noisy_mask = df[COL_DESC].apply(_is_noisy)
    df = df[~noisy_mask]
    logger.info("Noise filter removed %d rows.", noisy_mask.sum())

    if filter_language:
        logger.info("Running language detection (may take a moment) ...")
        df = df[df[COL_DESC].apply(_is_english)]
        logger.info("Language filter: %d English rows retained.", len(df))

    df = df.reset_index(drop=True)
    logger.info("Quality filtering complete -- %d rows remain.", len(df))
    return df


# ============================================================
# STEP 8 - SAMPLING
# ============================================================
def sample_data(
    df: pd.DataFrame,
    n: int = SAMPLE_SIZE,
    random_state: int = RANDOM_SEED,
) -> pd.DataFrame:
    """
    Draw a reproducible random sample.

    Parameters
    ----------
    df : pd.DataFrame
    n : int
    random_state : int

    Returns
    -------
    pd.DataFrame
    """
    if n >= len(df):
        logger.warning(
            "Sample size %d >= dataset size %d; returning full dataset.",
            n, len(df),
        )
        return df.reset_index(drop=True)
    sample = df.sample(n=n, random_state=random_state).reset_index(drop=True)
    logger.info("Sampled %d rows (seed=%d).", n, random_state)
    return sample


# ============================================================
# STEP 9 - EXPORT
# ============================================================
def save_to_csv(
    df: pd.DataFrame,
    filepath: str,
    encoding: str = "utf-8",
) -> None:
    """
    Save a DataFrame to CSV.

    Parameters
    ----------
    df : pd.DataFrame
    filepath : str
    encoding : str
    """
    try:
        df.to_csv(filepath, index=False, encoding=encoding)
        logger.info("Saved %d rows to '%s'.", len(df), filepath)
    except Exception as exc:
        logger.error("Failed to save '%s': %s", filepath, exc)
        raise


# ============================================================
# STEP 10 - RESEARCH STATISTICS (BONUS)
# ============================================================
def compute_stats(df: pd.DataFrame) -> None:
    """
    Print research-friendly summary statistics.

    Parameters
    ----------
    df : pd.DataFrame
    """
    print("=" * 50)
    print("RESEARCH STATISTICS")
    print("=" * 50)

    avg_len = df[COL_DESC].str.split().str.len().mean()
    print(f"Average description length  : {avg_len:.1f} words")

    if COL_SKILLS in df.columns:
        all_skills = [
            s.lower()
            for skill_list in df[COL_SKILLS]
            if isinstance(skill_list, list)
            for s in skill_list
        ]
        print(f"Unique skills across dataset: {len(set(all_skills))}")

    if COL_LEVEL in df.columns:
        print("\nDifficulty Distribution:")
        print(df[COL_LEVEL].value_counts().to_string())

    print("=" * 50)


# ============================================================
# MAIN PIPELINE
# ============================================================
def main() -> None:
    """Execute the full end-to-end preprocessing pipeline."""
    logger.info("=" * 55)
    logger.info("COURSERA NLP PREPROCESSING PIPELINE")
    logger.info("=" * 55)

    df_raw       = load_data()
    initial_size = len(df_raw)
    logger.info("Initial dataset size: %d rows.", initial_size)

    inspect_data(df_raw)
    print("\n--- 5 Sample Rows (raw) ---")
    display(df_raw.head(5))

    df = clean_data(df_raw)
    df = preprocess_text(df)
    df = normalize_features(df)
    df = deduplicate_data(df)
    df = filter_quality(df, filter_language=True)

    logger.info(
        "Pipeline summary -- Initial: %d | Final: %d | Removed: %d",
        initial_size, len(df), initial_size - len(df),
    )

    compute_stats(df)

    save_to_csv(df, OUTPUT_FULL_CSV)

    df_sample = sample_data(df, n=SAMPLE_SIZE)
    save_to_csv(df_sample, OUTPUT_SAMPLE_CSV)

    preview_cols = [
        c for c in [COL_TITLE, COL_LEVEL, COL_RATING,
                    COL_REVIEWS, COL_DESC]
        if c in df_sample.columns
    ]
    print("\n--- Final Sample Preview (5 rows) ---")
    display(df_sample[preview_cols].head(5))

    logger.info("=" * 55)
    logger.info("PIPELINE COMPLETE")
    logger.info("Outputs: %s | %s", OUTPUT_SAMPLE_CSV, OUTPUT_FULL_CSV)
    logger.info("=" * 55)

    try:
        from google.colab import files
        files.download(OUTPUT_SAMPLE_CSV)
        files.download(OUTPUT_FULL_CSV)
    except ImportError:
        logger.info("Files saved to current directory (not running in Colab).")


main()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/355 [00:00<?, ?B/s]

coursera_course_2024.csv:   0%|          | 0.00/23.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/6645 [00:00<?, ? examples/s]


--- 5 Sample Rows (raw) ---


...

,title,Level,rating,num_reviews,Description
0,cloud security basics,Beginner,4.4,990,this course introduces you to cybersecurity fo...
1,operational risk management frameworks strategies,Beginner,4.6,2920,in the final course from the risk management s...
2,navigating generative ai a ceo playbook,Unknown,4.8,720,this is a shortened executive summary of our c...
3,decoding ai a deep dive into ai models and pre...,Beginner,NaN,<NA>,decoding ai a deep dive into ai models and pre...
4,introduction to embedded machine learning,Intermediate,4.8,6630,machine learning ml allows us to teach compute...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>